# Student Performance Prediction System

## Exploratory Data Analysis (EDA)

### CodeVedX AI/ML Internship - Project 2

---

**Objective:**
This notebook performs a comprehensive Exploratory Data Analysis on the cleaned student performance dataset.
We analyse distributions, relationships, correlations, and outliers to uncover patterns
that inform the subsequent model training step.

**What this notebook covers:**
- Dataset overview and statistical summary
- Target variable (Exam_Score) distribution
- Univariate analysis: histograms and boxplots
- Bivariate analysis: scatter plots and grouped comparisons
- Correlation heatmap and matrix
- Categorical variable analysis
- Outlier detection (IQR method)
- Feature relationship summary
- Key observations and actionable insights

All charts are saved to outputs/charts/.


In [ ]:
# ========================================
# STEP 1: Import Libraries
# ========================================

import warnings
warnings.filterwarnings('ignore')

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

print('All libraries imported successfully')
print(f'  Python  : {sys.version.split()[0]}')
print(f'  Pandas  : {pd.__version__}')
print(f'  NumPy   : {np.__version__}')
print(f'  Matplotlib : {matplotlib.__version__}')
print(f'  Seaborn : {sns.__version__}')


In [ ]:
# ========================================
# STEP 2: Configure Display and Visual Style
# ========================================

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# High-quality publication-ready plots
plt.rcParams.update({
    'figure.dpi': 150,
    'figure.figsize': (10, 6),
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

PALETTE = sns.color_palette('viridis', 10)
sns.set_palette(PALETTE)

print('Display options and visual style configured.')


In [ ]:
# ========================================
# STEP 3: Setup Project Paths
# ========================================

PROJECT_ROOT = Path('..')
PROCESSED_DATA = PROJECT_ROOT / 'data' / 'processed' / 'student_performance.csv'
CHARTS_DIR = PROJECT_ROOT / 'outputs' / 'charts'
REPORTS_DIR = PROJECT_ROOT / 'outputs' / 'reports'

# Create directories if they do not exist
os.makedirs(CHARTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f'  Processed data : {PROCESSED_DATA}')
print(f'  Charts         : {CHARTS_DIR}')
print(f'  Reports        : {REPORTS_DIR}')


In [ ]:
# ========================================
# STEP 4: Load Processed Dataset
# ========================================

print('=' * 60)
print('LOADING PROCESSED DATASET')
print('=' * 60)

df = pd.read_csv(PROCESSED_DATA)

print(f"\nDataset loaded successfully")
print(f'  Source : {PROCESSED_DATA.name}')
print(f'  Rows   : {df.shape[0]:,}')
print(f'  Columns: {df.shape[1]}')


In [ ]:
# ========================================
# STEP 5: Dataset Overview
# ========================================

print('=' * 60)
print('DATASET OVERVIEW')
print('=' * 60)

print('\nFirst 5 rows:')
display(df.head())

print('\nDataset Information:')
print(f'  Memory  : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print(f'  Missing : {df.isnull().sum().sum()}')
print(f'  Dup rows: {df.duplicated().sum()}')


In [ ]:
# ========================================
# STEP 6: Statistical Summary
# ========================================

print('=' * 60)
print('STATISTICAL SUMMARY')
print('=' * 60)

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f'\nNumerical columns ({len(num_cols)}):')
print(num_cols)
display(df[num_cols].describe())

print(f'\nCategorical columns ({len(cat_cols)}):')
print(cat_cols)
display(df[cat_cols].describe())


In [ ]:
# ========================================
# STEP 7: Target Variable Distribution
# ========================================

print('=' * 60)
print('TARGET VARIABLE: EXAM SCORE')
print('=' * 60)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Histogram with KDE
sns.histplot(df['Exam_Score'], kde=True, bins=30, color=PALETTE[0], ax=axes[0])
axes[0].axvline(df['Exam_Score'].mean(), color='red', ls='--', lw=1.5, label=f'Mean={df["Exam_Score"].mean():.2f}')
axes[0].axvline(df['Exam_Score'].median(), color='green', ls=':', lw=1.5, label=f'Median={df["Exam_Score"].median():.2f}')
axes[0].set_title('Distribution of Exam Scores')
axes[0].set_xlabel('Exam Score')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# 2. Boxplot
sns.boxplot(y=df['Exam_Score'], color=PALETTE[1], ax=axes[1])
axes[1].set_title('Boxplot of Exam Scores')
axes[1].set_ylabel('Exam Score')

# 3. Q-Q plot for normality check
stats.probplot(df['Exam_Score'], dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot (Normality Check)')

plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'exam_score_distribution.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved: exam_score_distribution.png')

# Print target statistics
print(f'\n  Count   : {len(df):,}')
print(f'  Mean    : {df["Exam_Score"].mean():.2f}')
print(f'  Median  : {df["Exam_Score"].median():.2f}')
print(f'  Std Dev : {df["Exam_Score"].std():.2f}')
print(f'  Min     : {df["Exam_Score"].min():.0f}')
print(f'  Max     : {df["Exam_Score"].max():.0f}')
print(f'  Skewness: {df["Exam_Score"].skew():.4f}')
print(f'  Kurtosis: {df["Exam_Score"].kurtosis():.4f}')


In [ ]:
# ========================================
# STEP 8: Histograms - All Numerical Features
# ========================================

num_features = [c for c in num_cols if c != 'Exam_Score']

n_cols = 3
n_rows = int(np.ceil(len(num_features) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_features):
    sns.histplot(df[col], kde=True, bins=25, color=PALETTE[i % len(PALETTE)], ax=axes[i])
    axes[i].set_title(f'Distribution of {col}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'numerical_histograms.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved: numerical_histograms.png')


In [ ]:
# ========================================
# STEP 9: Boxplots - All Numerical Features
# ========================================

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_features):
    sns.boxplot(y=df[col], color=PALETTE[i % len(PALETTE)], ax=axes[i])
    axes[i].set_title(f'Boxplot of {col}')
    axes[i].set_ylabel(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'numerical_boxplots.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved: numerical_boxplots.png')


In [ ]:
# ========================================
# STEP 10: Correlation Heatmap
# ========================================

print('=' * 60)
print('CORRELATION HEATMAP')
print('=' * 60)

plt.figure(figsize=(12, 10))
corr_matrix = df[num_cols].corr()

# Mask upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='viridis',
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap - Numerical Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'correlation_heatmap.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved: correlation_heatmap.png')

# Top correlations with the target
target_corr = corr_matrix['Exam_Score'].drop('Exam_Score').sort_values(ascending=False)
print('\nCorrelation of features with Exam_Score (target):')
print(target_corr.to_string())


In [ ]:
# ========================================
# STEP 11: Attendance vs Exam Score
# ========================================

plt.figure(figsize=(10, 6))
sns.regplot(x=df['Attendance'], y=df['Exam_Score'],
            scatter_kws={'alpha': 0.3, 'color': PALETTE[0]},
            line_kws={'color': 'red', 'lw': 2})
plt.title('Attendance vs Exam Score', fontsize=14, fontweight='bold')
plt.xlabel('Attendance (%)')
plt.ylabel('Exam Score')
plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'attendance_vs_exam_score.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved: attendance_vs_exam_score.png')


In [ ]:
# ========================================
# STEP 12: Hours Studied vs Exam Score
# ========================================

plt.figure(figsize=(10, 6))
sns.regplot(x=df['Hours_Studied'], y=df['Exam_Score'],
            scatter_kws={'alpha': 0.3, 'color': PALETTE[1]},
            line_kws={'color': 'red', 'lw': 2})
plt.title('Hours Studied vs Exam Score', fontsize=14, fontweight='bold')
plt.xlabel('Hours Studied (per week)')
plt.ylabel('Exam Score')
plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'hours_studied_vs_exam_score.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved: hours_studied_vs_exam_score.png')


In [ ]:
# ========================================
# STEP 13: Previous Scores vs Exam Score
# ========================================

plt.figure(figsize=(10, 6))
sns.regplot(x=df['Previous_Scores'], y=df['Exam_Score'],
            scatter_kws={'alpha': 0.3, 'color': PALETTE[2]},
            line_kws={'color': 'red', 'lw': 2})
plt.title('Previous Scores vs Exam Score', fontsize=14, fontweight='bold')
plt.xlabel('Previous Scores')
plt.ylabel('Exam Score')
plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'previous_scores_vs_exam_score.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved: previous_scores_vs_exam_score.png')


In [ ]:
# ========================================
# STEP 14: Sleep Hours Analysis
# ========================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram
sns.histplot(df['Sleep_Hours'], kde=True, bins=15, color=PALETTE[3], ax=axes[0])
axes[0].set_title('Distribution of Sleep Hours')
axes[0].set_xlabel('Sleep Hours per Day')
axes[0].set_ylabel('Frequency')

# Boxplot
sns.boxplot(y=df['Sleep_Hours'], color=PALETTE[4], ax=axes[1])
axes[1].set_title('Boxplot of Sleep Hours')
axes[1].set_ylabel('Sleep Hours')

# Scatter vs Exam Score
sns.scatterplot(x=df['Sleep_Hours'], y=df['Exam_Score'], alpha=0.3, color=PALETTE[5], ax=axes[2])
axes[2].set_title('Sleep Hours vs Exam Score')
axes[2].set_xlabel('Sleep Hours')
axes[2].set_ylabel('Exam Score')

plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'sleep_hours_analysis.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved: sleep_hours_analysis.png')


In [ ]:
# ========================================
# STEP 15: Tutoring Sessions Analysis
# ========================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
sns.countplot(x=df['Tutoring_Sessions'], palette='viridis', ax=axes[0])
axes[0].set_title('Distribution of Tutoring Sessions')
axes[0].set_xlabel('Number of Tutoring Sessions')
axes[0].set_ylabel('Count')

# Boxplot vs Exam Score
sns.boxplot(x=df['Tutoring_Sessions'], y=df['Exam_Score'], palette='viridis', ax=axes[1])
axes[1].set_title('Exam Score by Tutoring Sessions')
axes[1].set_xlabel('Tutoring Sessions')
axes[1].set_ylabel('Exam Score')

plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'tutoring_sessions_analysis.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved: tutoring_sessions_analysis.png')


In [ ]:
# ========================================
# STEP 16: Categorical Variable Analysis
# ========================================

print('=' * 60)
print('CATEGORICAL VARIABLE ANALYSIS')
print('=' * 60)

n_cats = len(cat_cols)
fig, axes = plt.subplots(n_cats, 2, figsize=(16, n_cats * 4))

for i, col in enumerate(cat_cols):
    # Count plot (horizontal)
    order = df[col].value_counts().index
    sns.countplot(y=df[col], order=order, palette='viridis', ax=axes[i, 0])
    axes[i, 0].set_title(f'Count of {col}')
    axes[i, 0].set_xlabel('Count')
    axes[i, 0].set_ylabel(col)

    # Boxplot vs Exam Score
    sns.boxplot(x=df[col], y=df['Exam_Score'], palette='viridis', ax=axes[i, 1])
    axes[i, 1].set_title(f'Exam Score by {col}')
    axes[i, 1].set_xlabel(col)
    axes[i, 1].set_ylabel('Exam Score')
    axes[i, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(str(CHARTS_DIR / 'categorical_analysis.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved: categorical_analysis.png')

# Summary for each categorical feature
for col in cat_cols:
    print()
    print('=' * 50)
    print(f'{col}')
    print('=' * 50)
    print(f'  Unique values: {df[col].nunique()}')
    print(f'  Most common : {df[col].mode()[0]}')

    # Value counts
    print('  Value distribution:')
    for val, cnt in df[col].value_counts().items():
        print(f'    {str(val):25s}: {cnt:5d} ({cnt/len(df)*100:5.2f}%)')

    # Group mean of target
    group_means = df.groupby(col)['Exam_Score'].mean().sort_values(ascending=False)
    print('  Mean Exam Score by category:')
    for val, mean_val in group_means.items():
        print(f'    {str(val):25s}: {mean_val:.2f}')


In [ ]:
# ========================================
# STEP 17: Outlier Detection - IQR Method
# ========================================

print('=' * 60)
print('OUTLIER DETECTION (IQR Method)')
print('=' * 60)

outlier_records = []
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    pct = len(outliers) / len(df) * 100
    outlier_records.append({
        'Column': col,
        'Lower Fence': round(lower, 2),
        'Upper Fence': round(upper, 2),
        'Outlier Count': len(outliers),
        'Outlier %': round(pct, 2)
    })

outlier_df = pd.DataFrame(outlier_records)
display(outlier_df)

# Print columns with outliers
print('\nColumns with detected outliers:')
has_outliers = outlier_df[outlier_df['Outlier Count'] > 0]
if len(has_outliers) > 0:
    for _, row in has_outliers.iterrows():
        print(f'  {row["Column"]:25s}: {int(row["Outlier Count"]):4d} outliers ({row["Outlier %"]:.2f}%)')
else:
    print('  No outliers detected.')


In [ ]:
# ========================================
# STEP 18: Outlier Visualization (Boxplots)
# ========================================

# Select up to 6 numerical features to visualize
cols_to_plot = []
for c in num_features:
    count_val = outlier_df[outlier_df['Column'] == c]['Outlier Count'].values[0]
    if count_val > 0:
        cols_to_plot.append(c)
cols_to_plot = cols_to_plot[:6]

if len(cols_to_plot) > 0:
    n_plot_cols = min(3, len(cols_to_plot))
    n_plot_rows = int(np.ceil(len(cols_to_plot) / n_plot_cols))
    fig, axes = plt.subplots(n_plot_rows, n_plot_cols, figsize=(15, n_plot_rows * 4))
    axes = axes.flatten()

    for i, col in enumerate(cols_to_plot):
        sns.boxplot(y=df[col], color=PALETTE[i], ax=axes[i])
        axes[i].set_title(f'{col}')
        axes[i].set_ylabel('')

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Outlier Detection - Boxplots', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(CHARTS_DIR / 'outlier_boxplots.png'), bbox_inches='tight', dpi=150)
    plt.show()
    print('Chart saved: outlier_boxplots.png')
else:
    print('No columns with outliers to plot.')


In [ ]:
# ========================================
# STEP 19: Pair Plot - Key Features
# ========================================

print('=' * 60)
print('PAIR PLOT - KEY FEATURES')
print('=' * 60)

key_features = ['Exam_Score', 'Attendance', 'Hours_Studied', 'Previous_Scores',
                'Sleep_Hours', 'Tutoring_Sessions']
available = [c for c in key_features if c in df.columns]

sns.pairplot(df[available], diag_kind='kde', palette='viridis',
             plot_kws={'alpha': 0.2, 's': 10})
plt.suptitle('Pair Plot - Key Numerical Features', y=1.02, fontsize=16, fontweight='bold')
plt.savefig(str(CHARTS_DIR / 'pairplot_key_features.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Chart saved: pairplot_key_features.png')


In [ ]:
# ========================================
# STEP 20: Feature Relationships Summary
# ========================================

print('=' * 60)
print('FEATURE RELATIONSHIPS SUMMARY')
print('=' * 60)

target = 'Exam_Score'
relationship_data = []

for col in num_cols:
    if col == target:
        continue
    r = corr_matrix.loc[col, target]
    strength = 'Strong' if abs(r) >= 0.5 else ('Moderate' if abs(r) >= 0.3 else 'Weak')
    direction = 'Positive' if r > 0 else 'Negative'
    relationship_data.append({
        'Feature': col,
        'Correlation': round(r, 4),
        'Strength': strength,
        'Direction': direction
    })

rel_df = pd.DataFrame(relationship_data).sort_values('Correlation', ascending=False)
print('\nCorrelation with Exam_Score:')
display(rel_df)

print('\nKey Insights:')
print(f'  - Best positive predictor: {rel_df.iloc[0]["Feature"]}  (r = {rel_df.iloc[0]["Correlation"]:.4f})')
print(f'  - Best negative predictor: {rel_df.iloc[-1]["Feature"]} (r = {rel_df.iloc[-1]["Correlation"]:.4f})')
print('  - Features with moderate or strong correlation (|r| >= 0.3):')
strong = rel_df[abs(rel_df['Correlation']) >= 0.3]
for _, row in strong.iterrows():
    print(f'      {row["Feature"]:25s}: {row["Correlation"]:.4f} ({row["Direction"]}, {row["Strength"]})')


In [ ]:
# ========================================
# STEP 21: Important EDA Observations
# ========================================

print('=' * 60)
print('KEY EDA OBSERVATIONS')
print('=' * 60)

# Dynamically generate observations
best_feature = rel_df.iloc[0]['Feature']
best_corr = rel_df.iloc[0]['Correlation']
worst_feature = rel_df.iloc[-1]['Feature']
worst_corr = rel_df.iloc[-1]['Correlation']

observations = [
    f'1. Dataset contains {len(df):,} records with {len(num_cols)} numerical and {len(cat_cols)} categorical features.',
    f'2. Target (Exam_Score) ranges from {df["Exam_Score"].min():.0f} to {df["Exam_Score"].max():.0f} '
    f'with mean = {df["Exam_Score"].mean():.2f} and median = {df["Exam_Score"].median():.2f}.',
    f'3. Target distribution is approximately {"normal" if abs(df["Exam_Score"].skew()) < 0.5 else "skewed"} '
    f'(skewness = {df["Exam_Score"].skew():.4f}).',
    f'4. {best_feature} has the strongest {"positive" if best_corr > 0 else "negative"} correlation '
    f'with Exam_Score (r = {best_corr:.4f}).',
    f'5. {worst_feature} has the weakest correlation (r = {worst_corr:.4f}).',
    '6. Several categorical variables (e.g. Access_to_Resources, Parental_Involvement, Motivation_Level) '
    'show meaningful variance in mean Exam_Score across categories.',
    f'7. Outlier analysis shows minimal outliers across features '
    f'(max {outlier_df["Outlier %"].max():.2f}%). No extreme data quality issues.',
    '8. No missing values or duplicates remain after preprocessing (Notebook 1).',
    '9. Next step: Encode categorical features, split data, and train regression models.'
]

for obs in observations:
    print(f'\n  {obs}')


In [ ]:
# ========================================
# STEP 22: EDA Summary
# ========================================

print('=' * 60)
print('EDA SUMMARY')
print('=' * 60)

import glob
charts = sorted(CHARTS_DIR.glob('*.png'))
print(f'\nCharts generated: {len(charts)}')
for c in charts:
    print(f'  {c.name}')

print(f'\n{"=" * 60}')
print('  EXPLORATORY DATA ANALYSIS COMPLETED SUCCESSFULLY')
print(f'{"=" * 60}')
print('\nClean dataset verified')
print('Target distribution understood')
print('Feature relationships identified')
print('Key insights documented')
print('Charts saved for reporting')
print('\nProceed to Notebook 3: Model Training')
